# Gold — dim_geografia

Construye la dimensión geográfica unificando los tres niveles de agregación del DW.

| Columna | Tipo | Descripción |
|---|---|---|
| `geografia_sk` | INT | Surrogate key |
| `pais` | STRING | País |
| `departamento` | STRING | Departamento (NULL si nivel=pais sin desglose, 'Sin desagregar' para INE-edad) |
| `municipio` | STRING | Municipio (NULL si nivel != municipio) |
| `nivel_geo` | STRING | `pais` \| `departamento` \| `municipio` |

**Fila obligatoria:** `(pais='Guatemala', departamento='Sin desagregar', municipio=NULL, nivel_geo='pais')` — apunta a ella `fact_defunciones` para las tablas INE-edad (sin dimensión geográfica).

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, StructType, StructField
from functools import reduce
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

SILVER_SCHEMA = 'stage_silver_ss2'
GOLD_SCHEMA   = 'gold_ss2'
WRITE_MODE    = 'overwrite'

_GEO_SCHEMA = StructType([
    StructField('pais',         StringType(), True),
    StructField('departamento', StringType(), True),
    StructField('municipio',    StringType(), True),
    StructField('nivel_geo',    StringType(), True),
])

def get_job_run_id():
    try:
        return (
            dbutils.notebook.entry_point
            .getDbutils().notebook().getContext()
            .currentRunId().toString()
        )
    except Exception:
        return 'manual'

RUN_ID = get_job_run_id()
logger.info(f'Setup OK — run_id={RUN_ID}')

## Nivel PAIS — fuentes WHO

Lee los países distintos de las tablas Silver WHO. Produce filas con `departamento=NULL`.

In [ ]:
df_who_gt = spark.table(f'{SILVER_SCHEMA}.who_guatemala').select('pais')
df_who_cr = spark.table(f'{SILVER_SCHEMA}.who_costa_rica').select('pais')

df_pais_who = (
    df_who_gt.union(df_who_cr)
    .distinct()
    .withColumn('pais', F.initcap(F.regexp_replace(F.col('pais'), '_', ' ')))
    .withColumn('departamento', F.lit(None).cast(StringType()))
    .withColumn('municipio',    F.lit(None).cast(StringType()))
    .withColumn('nivel_geo',    F.lit('pais'))
)
logger.info(f'Nivel pais (WHO): {df_pais_who.count()} filas')

## Nivel PAIS — placeholder INE-edad

Las tablas INE-edad no tienen columna de departamento. Sus filas en `fact_defunciones`
apuntan a esta fila placeholder para indicar que la geografía no está desagregada.

In [ ]:
df_pais_placeholder = spark.createDataFrame(
    [('Guatemala', 'Sin desagregar', None, 'pais')],
    _GEO_SCHEMA
)
logger.info('Placeholder INE-edad creado: (Guatemala, Sin desagregar, NULL, pais)')

## Nivel DEPARTAMENTO — INE-geo + MSPAS

Lee `departamento_oficial` de las 5 tablas Silver con dimensión departamental.

In [ ]:
_DEPTO_TABLES = [
    'ine_defunciones_depto_residencia',
    'ine_defunciones_causas_externas',
    'dbx_primeras_causas_de_morbilidad_2015_a_2025',
    'dbx_morbilidad_enfermedades_cronicas_2015_a_2025',
    'dbx_morbilidad_grupo_materno_infantil_2012_a_2025',
]

depto_dfs = [
    spark.table(f'{SILVER_SCHEMA}.{t}').select(F.col('departamento_oficial').alias('departamento'))
    for t in _DEPTO_TABLES
]

df_depto = (
    reduce(lambda a, b: a.union(b), depto_dfs)
    .distinct()
    .filter(F.col('departamento').isNotNull())
    .filter(~F.col('departamento').isin(['Todos los departamentos']))
    .withColumn('pais',      F.lit('Guatemala'))
    .withColumn('municipio', F.lit(None).cast(StringType()))
    .withColumn('nivel_geo', F.lit('departamento'))
)
logger.info(f'Nivel departamento: {df_depto.count()} deptos únicos')

## Nivel MUNICIPIO — MSPAS

Lee combinaciones únicas `(departamento_oficial, municipio)` de las 3 tablas MSPAS.

In [ ]:
_MSPAS_TABLES = [
    'dbx_primeras_causas_de_morbilidad_2015_a_2025',
    'dbx_morbilidad_enfermedades_cronicas_2015_a_2025',
    'dbx_morbilidad_grupo_materno_infantil_2012_a_2025',
]

muni_dfs = [
    spark.table(f'{SILVER_SCHEMA}.{t}')
    .select(
        F.col('departamento_oficial').alias('departamento'),
        F.col('municipio')
    )
    for t in _MSPAS_TABLES
]

df_muni = (
    reduce(lambda a, b: a.union(b), muni_dfs)
    .distinct()
    .filter(F.col('departamento').isNotNull())
    .filter(F.col('municipio').isNotNull())
    .withColumn('pais',      F.lit('Guatemala'))
    .withColumn('nivel_geo', F.lit('municipio'))
)
logger.info(f'Nivel municipio: {df_muni.count()} combinaciones depto+municipio')

## UNION y escritura

In [ ]:
_COLS = ['pais', 'departamento', 'municipio', 'nivel_geo']

df_all = (
    df_pais_who.select(*_COLS)
    .union(df_pais_placeholder.select(*_COLS))
    .union(df_depto.select(*_COLS))
    .union(df_muni.select(*_COLS))
    .dropDuplicates()
)

df_dim = df_all.withColumn(
    'geografia_sk',
    F.row_number().over(Window.orderBy('nivel_geo', 'pais', 'departamento', 'municipio'))
)

df_dim = df_dim.select('geografia_sk', 'pais', 'departamento', 'municipio', 'nivel_geo')

logger.info(f'dim_geografia: {df_dim.count()} filas')

df_dim.write \
    .format('delta') \
    .mode(WRITE_MODE) \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{GOLD_SCHEMA}.dim_geografia')

logger.info(f'Escrito → {GOLD_SCHEMA}.dim_geografia')

## Validación

In [ ]:
df_val = spark.table(f'{GOLD_SCHEMA}.dim_geografia')
print(f'Total filas: {df_val.count()}')

print('\n── Distribución por nivel_geo ──')
df_val.groupBy('nivel_geo').count().orderBy('nivel_geo').show()

print('\n── Filas nivel PAIS ──')
df_val.filter(F.col('nivel_geo') == 'pais').show(truncate=False)

print('\n── Verificar placeholder INE-edad ──')
placeholder = df_val.filter(
    (F.col('pais') == 'Guatemala') & (F.col('departamento') == 'Sin desagregar')
).count()
print(f'Placeholder (Guatemala / Sin desagregar): {placeholder} fila (debe ser 1)')

print('\n── Muestra departamentos ──')
df_val.filter(F.col('nivel_geo') == 'departamento').show(25, truncate=False)